# Time-Series-to-Image Transformations

This notebook demonstrates all six transformation methods applied to BCI Competition IV-2a EEG data.

**Transformations:**
1. GAF (Gramian Angular Field)
2. MTF (Markov Transition Field)
3. Recurrence Plot
4. Spectrogram (STFT)
5. Scalogram (CWT)
6. Topographic (SSFI)

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import h5py
from src.transforms import (
    GAFTransformer,
    MTFTransformer,
    RecurrencePlotTransformer,
    SpectrogramTransformer,
    ScalogramTransformer,
    TopographicTransformer,
    get_transformer,
    list_transformers
)

# Setup plotting
plt.style.use('default')
%matplotlib inline

## 1. Load Preprocessed Data

Load a sample epoch from preprocessed BCI IV-2a data.

In [ ]:
# Load preprocessed data
preprocessed_file = Path('../data/preprocessed/bci_iv_2a/A01T_preprocessed.h5')

with h5py.File(preprocessed_file, 'r') as f:
    data = f['data'][:]
    labels = f['labels'][:]
    ch_names = eval(f.attrs['ch_names'])
    sfreq = f.attrs['sfreq']

print(f"Data shape: {data.shape}")  # (n_epochs, n_channels, n_times)
print(f"Channels: {len(ch_names)}")
print(f"Sampling frequency: {sfreq} Hz")
print(f"Labels: {np.unique(labels)} (class counts: {np.bincount(labels)})")

# Select one epoch from each class for visualization
class_names = ['Left Hand', 'Right Hand', 'Feet', 'Tongue']
sample_epochs = {}
for class_id in range(4):
    idx = np.where(labels == class_id)[0][0]  # First epoch of this class
    sample_epochs[class_names[class_id]] = data[idx]
    print(f"  {class_names[class_id]}: epoch {idx}, shape {data[idx].shape}")

## 2. Visualize Raw EEG Time-Series

Plot the raw EEG signals before transformation.

In [ ]:
# Plot raw EEG for one epoch
epoch = sample_epochs['Left Hand']
time = np.arange(epoch.shape[1]) / sfreq

fig, axes = plt.subplots(5, 5, figsize=(15, 12))
axes = axes.flatten()

for i in range(min(25, epoch.shape[0])):
    axes[i].plot(time, epoch[i], linewidth=0.5)
    axes[i].set_title(f'{ch_names[i]}', fontsize=8)
    axes[i].set_xlim(0, time[-1])
    axes[i].tick_params(labelsize=6)
    if i >= 20:
        axes[i].set_xlabel('Time (s)', fontsize=7)
    if i % 5 == 0:
        axes[i].set_ylabel('Amplitude (µV)', fontsize=7)

plt.suptitle('Raw EEG Time-Series (Left Hand Motor Imagery)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## 3. GAF Transformation

Gramian Angular Field encodes time-series in polar coordinates.

In [ ]:
# Transform using GAF (Summation)
gaf_transformer = GAFTransformer(image_size=64, method='summation')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (class_name, epoch) in enumerate(sample_epochs.items()):
    # Transform epoch - take first channel only for visualization
    gaf_images = gaf_transformer.transform_epoch(epoch)
    
    # Display first channel
    im = axes[idx].imshow(gaf_images[0], cmap='RdBu_r', aspect='auto')
    axes[idx].set_title(f'GAF - {class_name}\n(Channel: {ch_names[0]})')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Time')
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle('Gramian Angular Field (GASF) Transformation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"GAF output shape: {gaf_images.shape}")  # (n_channels, 64, 64)

## 4. MTF Transformation

Markov Transition Field captures temporal transition probabilities.

In [ ]:
# Transform using MTF
mtf_transformer = MTFTransformer(image_size=64, n_bins=8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (class_name, epoch) in enumerate(sample_epochs.items()):
    # Transform epoch - first channel only
    mtf_images = mtf_transformer.transform_epoch(epoch)
    
    im = axes[idx].imshow(mtf_images[0], cmap='viridis', aspect='auto')
    axes[idx].set_title(f'MTF - {class_name}\n(Channel: {ch_names[0]})')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Time')
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle('Markov Transition Field (Q=8) Transformation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"MTF output shape: {mtf_images.shape}")

## 5. Recurrence Plot Transformation

Recurrence Plots reveal hidden patterns in phase space.

In [ ]:
# Transform using Recurrence Plot
rp_transformer = RecurrencePlotTransformer(
    embedding_dim=3,
    time_delay=1,
    threshold_percentile=10.0
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (class_name, epoch) in enumerate(sample_epochs.items()):
    # Transform epoch - first channel only
    rp_images = rp_transformer.transform_epoch(epoch, target_size=64)
    
    im = axes[idx].imshow(rp_images[0], cmap='binary', aspect='auto')
    axes[idx].set_title(f'RP - {class_name}\n(Channel: {ch_names[0]})')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Time')
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle('Recurrence Plot (m=3, τ=1) Transformation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"RP output shape: {rp_images.shape}")

## 6. Spectrogram Transformation

STFT Spectrogram shows time-frequency content.

In [ ]:
# Transform using Spectrogram
spec_transformer = SpectrogramTransformer(
    sfreq=250.0,
    window_length=0.5,
    overlap=0.5,
    freq_range=(1.0, 50.0)
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (class_name, epoch) in enumerate(sample_epochs.items()):
    # Transform epoch - first channel only
    spec_images = spec_transformer.transform_epoch(epoch, target_size=64)
    
    im = axes[idx].imshow(spec_images[0], cmap='jet', aspect='auto', origin='lower')
    axes[idx].set_title(f'Spectrogram - {class_name}\n(Channel: {ch_names[0]})')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Frequency')
    plt.colorbar(im, ax=axes[idx], fraction=0.046, label='Power')

plt.suptitle('STFT Spectrogram Transformation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Spectrogram output shape: {spec_images.shape}")

## 7. Scalogram Transformation

CWT Scalogram provides multi-resolution time-frequency analysis.

In [ ]:
# Transform using Scalogram
cwt_transformer = ScalogramTransformer(
    sfreq=250.0,
    freq_range=(1.0, 50.0),
    n_scales=64,
    wavelet='morl'
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (class_name, epoch) in enumerate(sample_epochs.items()):
    # Transform epoch - first channel only
    cwt_images = cwt_transformer.transform_epoch(epoch, target_size=64)
    
    im = axes[idx].imshow(cwt_images[0], cmap='jet', aspect='auto', origin='lower')
    axes[idx].set_title(f'Scalogram - {class_name}\n(Channel: {ch_names[0]})')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Scale (Frequency)')
    plt.colorbar(im, ax=axes[idx], fraction=0.046, label='Power')

plt.suptitle('CWT Scalogram (Morlet) Transformation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Scalogram output shape: {cwt_images.shape}")

## 8. Topographic Transformation

SSFI creates spatial maps preserving electrode locations.

In [ ]:
# Transform using Topographic (SSFI)
topo_transformer = TopographicTransformer(
    ch_names=ch_names,
    grid_size=64,
    sfreq=250.0
)

# Transform one epoch
epoch = sample_epochs['Left Hand']
topo_images = topo_transformer.transform_epoch_ssfi(epoch)

# Display different frequency bands
band_names = list(topo_transformer.bands.keys())
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, band_name in enumerate(band_names):
    im = axes[idx].imshow(topo_images[idx], cmap='RdBu_r', aspect='auto')
    band_range = topo_transformer.bands[band_name]
    axes[idx].set_title(f'{band_name.capitalize()}\n({band_range[0]}-{band_range[1]} Hz)')
    axes[idx].axis('off')
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle('Topographic Maps - Left Hand Motor Imagery (Multi-Band)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print(f"Topographic output shape: {topo_images.shape}")  # (n_bands, grid_size, grid_size)

## 9. Compare All Transformations

Side-by-side comparison of all methods on the same channel.

In [ ]:
# Transform using all methods
epoch = sample_epochs['Left Hand']

transforms = {
    'GAF': gaf_transformer.transform_epoch(epoch)[0],
    'MTF': mtf_transformer.transform_epoch(epoch)[0],
    'RP': rp_transformer.transform_epoch(epoch, target_size=64)[0],
    'Spectrogram': spec_transformer.transform_epoch(epoch, target_size=64)[0],
    'Scalogram': cwt_transformer.transform_epoch(epoch, target_size=64)[0],
    'Topo (Alpha)': topo_images[2]  # Alpha band
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, image) in enumerate(transforms.items()):
    if 'Spec' in name or 'Scalo' in name:
        im = axes[idx].imshow(image, cmap='jet', aspect='auto', origin='lower')
    elif 'RP' in name:
        im = axes[idx].imshow(image, cmap='binary', aspect='auto')
    else:
        im = axes[idx].imshow(image, cmap='RdBu_r', aspect='auto')
    
    axes[idx].set_title(name, fontsize=12)
    axes[idx].axis('off')
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle(f'All Transformations - Left Hand Motor Imagery\n(Channel: {ch_names[0]})', 
             fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

## 10. Load and Visualize Saved Transformed Images

Load previously saved HDF5 files and display statistics.

In [ ]:
# Load saved transformations
image_files = {
    'GAF': '../data/images/gaf/A01T_gaf_summation.h5',
    'MTF': '../data/images/mtf/A01T_mtf_q8.h5',
    'Spectrogram': '../data/images/spec/A01T_spec.h5',
}

for name, file_path in image_files.items():
    if Path(file_path).exists():
        with h5py.File(file_path, 'r') as f:
            images = f['images'][:]
            labels = f['labels'][:]
            transform_type = f.attrs.get('transform', 'unknown')
            
            print(f"\n{name} ({transform_type}):")
            print(f"  Shape: {images.shape}")
            print(f"  Size: {Path(file_path).stat().st_size / 1024**2:.2f} MB")
            print(f"  Value range: [{images.min():.3f}, {images.max():.3f}]")
            print(f"  Labels: {np.unique(labels)} (counts: {np.bincount(labels)})")
    else:
        print(f"\n{name}: File not found")

## 11. Transform Registry Demo

Demonstrate the transform registry for easy instantiation.

In [ ]:
# List all available transformers
print("Available transformers:")
for t in list_transformers():
    print(f"  - {t}")

# Get transformer from registry
print("\nInstantiating transformers from registry:")
transformer1 = get_transformer('gaf_summation', image_size=128)
print(f"  {transformer1.__class__.__name__}: image_size={transformer1.image_size}, method={transformer1.method}")

transformer2 = get_transformer('mtf_q16', image_size=64)
print(f"  {transformer2.__class__.__name__}: image_size={transformer2.image_size}, n_bins={transformer2.n_bins}")

transformer3 = get_transformer('cwt_morlet', sfreq=250.0, n_scales=64)
print(f"  {transformer3.__class__.__name__}: n_scales={transformer3.n_scales}, wavelet={transformer3.wavelet}")

## Summary

This notebook demonstrated all six time-series-to-image transformations:

1. **GAF**: Polar coordinate encoding preserving temporal correlations
2. **MTF**: Transition probability matrices capturing state changes
3. **Recurrence Plot**: Phase space patterns revealing nonlinear dynamics
4. **Spectrogram**: Classical time-frequency representation via STFT
5. **Scalogram**: Multi-resolution time-frequency via continuous wavelets
6. **Topographic**: Spatial maps preserving electrode locations and frequency bands

Each transformation provides a unique perspective on the EEG data, suitable for training deep learning models.

**Next Steps:**
- Generate images for all subjects using `experiments/scripts/transform_all_bci_iv_2a.py`
- Implement CNN and ViT models (Phase 4)
- Train and evaluate classifiers (Phases 5-6)